# Comp1 - Radiomics

This notebook supports conventional radiomics modeling and characterization, for example evaluating the contribution of rad_score to clinical diagnosis.

The expected data structure is as follows (file and directory names may vary):
1. `images`: CT, MRI, or other images for all study participants.
2. `masks`: manually delineated ROIs, with one mask corresponding to each image in `images`.
3. `label.txt`: patient labels, such as benign/malignant status or 5-year survival status.

## Onekey workflow

1. Validate the data format.
2. Extract radiomics features after data validation passes.
3. Read the label data.
4. Merge features with labels to construct the dataset.
5. Inspect summary statistics and check for outliers.
6. Standardize the data to zero mean and unit variance.
7. Select features using correlation coefficients, such as Spearman or Pearson correlation.
8. Construct training and test sets. This example uses a random split; prepare appropriate independent datasets for multicenter validation.
9. Select features with nonzero LASSO coefficients for subsequent modeling.
10. Train machine learning models, such as LR, SVM, and RF.
11. Visualize model performance, including AUC, ROC curves, and confusion matrices.


In [ ]:
## Video tutorials
import os
os.environ['DISABLE_VIDEO'] = 'TRUE'
from onekey_algo.custom.Manager import onekey_show
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1')

## 1. Data validation
Validate the diagnostic data first. Continue when validation passes; otherwise correct the data according to the reported issues.

**Note**: Filenames in `images` and `masks` must match exactly. For example, `images/1.nii.gz` must have a corresponding `masks/1.nii.gz`.

A custom function may also be used to load and parse the data.

In [ ]:
# Data validation tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u6570\u636e\u68c0\u9a8c')

### Specify input data

Define the following three parameters:

1. `mydir`: path to the data directory.
2. `labelf`: file containing sample labels.
3. `labels`: prediction targets, such as benign/malignant status or T stage.

In [ ]:
import os
import pandas as pd
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('features', exist_ok=True)

# Set the task prefix
task_type = 'HabitatPre_'
# Set the data directory
# mydir = r'path/to/your/data'
mydir = get_param_in_cwd('radio_dir')
if mydir == okds.ct:
    print(f'Using Onekey data: {okds.ct}; update the data directory if this is not the intended dataset.')
# Corresponding label file
group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file') or os.path.join(mydir, 'label.csv')
# Read label column names
labels = [get_param_in_cwd('task_column') or 'label']

### Match images and masks

Filenames in `images` and `masks` must match exactly. For example, `images/1.nii.gz` must have a corresponding `masks/1.nii.gz`.

A custom function may also be used to load and parse the data.

In [ ]:
from pathlib import Path
from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir

for modal in get_param_in_cwd('modals'):
    images, masks = get_image_mask_from_dir(mydir, images=f'{modal}/images_re', masks=f'{modal}/habitat/cluster_3')
    # diagnose_3d_image_mask_settings(images, masks, verbose=False)
    print(f'Loaded {len(images)} samples.')

# Conventional radiomics features

Extract conventional radiomics features using PyRadiomics. This section usually requires no changes. The Onekey wrapper accepts the following arguments.

```python
def extract(self, images: Union[str, List[str]], 
            masks: Union[str, List[str]], labels: Union[int, List[int]] = 1, settings=None)
"""
    * images: list of images for feature extraction.
    * masks: list of masks corresponding one-to-one with images.
    * labels: segmentation labels for feature extraction; defaults to label=1.
    * settings: additional extraction parameters; defaults to None.

"""
```

```python
def get_label_data_frame(self, label: int = 1, column_names=None, images='images', masks='labels')
"""
    * label: segmentation label for which to extract features.
    * columns_names: defaults to None to use the predefined column names.
"""
```
    
```python
def get_image_mask_from_dir(root, images='images', masks='labels')
"""
    * root: directory containing data for feature extraction.
    * images: name of the image subdirectory under root.
    * masks: name of the mask subdirectory under root.
"""
```


In [ ]:
# Feature extraction tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u7279\u5f81\u63d0\u53d6')

In [ ]:
import warnings
import pandas as pd
import numpy as np
from onekey_algo.custom.components.Radiology import ConventionalRadiomics
from onekey_algo.custom.components.habitat import habitat_feature_fusion
 
warnings.filterwarnings("ignore")

rad_data = []
habitats = [1, 2, 3]
param_file = get_param_in_cwd('extractor_settings')
for modal in get_param_in_cwd('modals'):
    rad_ = []
    for habitat in habitats:
        if os.path.exists(f'features/h{habitat}_rad_features_{modal}.csv'):
            rad_data_ = pd.read_csv(f'features/h{habitat}_rad_features_{modal}.csv', header=0)
        else:
            images, masks = get_image_mask_from_dir(mydir, images=f'{modal}/images_re', masks=f'{modal}/habitat/cluster_3')
            radiomics = ConventionalRadiomics(param_file, correctMask=True)
            radiomics.extract(images, masks, workers=3, labels=habitat)
            rad_data_ = radiomics.get_label_data_frame(label=habitat)
            rad_data_.columns = [f"{c.replace('-', '_')}_h{habitat}" if c != 'ID' else 'ID' for c in rad_data_.columns]
            rad_data_.to_csv(f'features/h{habitat}_rad_features_{modal}.csv', header=True, index=False)
        rad_.append(rad_data_)
    if os.path.exists(f'features/rad_features_habitat_{modal}.csv') and False:
        habitat_data = pd.read_csv(f'features/rad_features_habitat_{modal}.csv')
    else:
        habitat_data = habitat_feature_fusion(*rad_, mode='fill')
        habitat_data.to_csv(f'features/rad_features_habitat_{modal}.csv', index=False)
    habitat_data.columns = [f"{c}_{modal}" if c != 'ID' else 'ID' for c in habitat_data.columns]
    rad_data.append(habitat_data)

rad_data, post = rad_data
rad_data

## Feature statistics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sorted_counts = pd.DataFrame([c.split('_')[-4] for c in rad_data.columns if c !='ID']).value_counts()
sorted_counts = pd.DataFrame(sorted_counts, columns=['count']).reset_index()
sorted_counts = sorted_counts.sort_values(0)
sorted_counts.columns = ['feature_group', 'count']
display(sorted_counts)

plt.figure(figsize=(20, 10))
ax = plt.subplot(121)
plt.pie(sorted_counts['count'], labels=[i for i in sorted_counts['feature_group']], startangle=0,
        counterclock = False, autopct = '%.1f%%')
# plt.bar_label(bar.containers[0])
ax = plt.subplot(122)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
bar = sns.barplot(data=sorted_counts, x='feature_group', y='count', )
plt.savefig(f'img/{task_type}feature_ratio.svg', bbox_inches = 'tight')

## Label data

Data are stored in CSV format. Use a custom reader for other formats.

`label_data` must be a `DataFrame` with an ID column followed by one or more label columns; multiple targets are supported.

In [ ]:
group_info = 'group'
label_data = pd.read_csv(labelf)
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
# g = pd.read_csv('ids.csv')
# label_data = pd.merge(label_data, g, on='ID', how='inner')
label_data = label_data[['ID', group_info] + labels]
label_data

## Merge features and labels 

Merge `label_data` with `rad_data` to construct the training data.

**Note: ** 
1. Remove the ID column from the model features.
2. If samples are missing after merging, check that the identifiers match.

In [ ]:
# Remove the ID column.
from onekey_algo.custom.utils import print_join_info
print_join_info(rad_data, label_data)
combined_data = pd.merge(rad_data, label_data, on=['ID'], how='inner')
# combined_data[['ID'] + selected_features[0]].to_csv('features/sel_habitat.csv', index=False)
ids = combined_data['ID']
combined_data = combined_data.drop(['ID'], axis=1)
display(combined_data[labels].value_counts())
display(combined_data[group_info].value_counts())
combined_data

## Inspect summary statistics

1. count: number of samples.
2. mean and std: feature mean and standard deviation.
3. min, 25%, 50%, 75%, max: minimum, quartiles, and maximum of each feature.

In [ ]:
combined_data.describe()

## Standardization

Onekey's `normalize_df` standardizes data to zero mean and unit variance using the following transformation:

$column = \frac{column - mean}{std}$

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df
data = normalize_df(combined_data, not_norm=labels, group=group_info, use_train=True)
data = data.dropna(axis=1)
data.to_csv(f'features/{task_type}feature_norm.csv', index=False)
data.describe()

## Statistical tests

Select features using a t-test or Mann-Whitney U test.

**Note:** This optional step depends on the study design. Adjust `pvalue` as needed; the default significance threshold is 0.05.

In [ ]:
import seaborn as sns
from onekey_algo.custom.components.stats import clinic_stats

sub_data = data[data[group_info] == 'train']
stats = clinic_stats(sub_data, stats_columns=list(data.columns[0:-2]), label_column=labels[0], 
                     continuous_columns=list(data.columns[0:-2]))
stats

#### Plot feature distributions

In [ ]:
import matplotlib.pyplot as plt

def map2float(x):
    try:
        return float(str(x)[1:])
    except:
        return 1

stats[['pvalue']] = stats[['pvalue']].applymap(map2float)
stats[['group']] = stats[['feature_name']].applymap(lambda x: x.split('_')[-4])
stats = stats[['feature_name', 'pvalue', 'group']]
g = sns.catplot(x="group", y="pvalue", data=stats, kind="violin")
g.fig.set_size_inches(15,10)
sns.stripplot(x="group", y="pvalue", data=stats, ax=g.ax, color='black')
plt.savefig(f'img/{task_type}feature_stats.svg', bbox_inches = 'tight')

#### Adjust pvalue for feature selection.

In [ ]:
pvalue = 0.05
sel_feature = list(stats[stats['pvalue'] < pvalue]['feature_name']) + labels + [group_info]
sub_data = sub_data[sel_feature]
sub_data

### Correlation coefficient

Three correlation methods are available:
1. pearson: Pearson correlation coefficient

2. kendall: Kendall Tau correlation coefficient

3. spearman: Spearman rank correlation

Reference for the three correlation methods: https://blog.csdn.net/zmqsdu9001/article/details/82840332

In [ ]:
# Correlation tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u76f8\u5173\u7cfb\u6570')

In [ ]:
pearson_corr = sub_data[[c for c in sub_data.columns if c not in labels]].corr('pearson')
# kendall_corr = data[[c for c in data.columns if c not in labels]].corr('kendall')
# spearman_corr = data[[c for c in data.columns if c not in labels]].corr('spearman')

### Visualize correlations

Change the selected variable to visualize matrices for different correlation methods.

**Note:** Avoid this visualization when there are more than 100 features, as it can be slow.

In [ ]:
import seaborn as sns
from onekey_algo.custom.components.comp1 import draw_matrix

if data.shape[1] < 100:
    plt.figure(figsize=(50.0, 40.0))

    # Select the correlation method to visualize
    draw_matrix(pearson_corr, annot=True, cmap='YlGnBu', cbar=False)
    plt.savefig(f'img/{task_type}Rad_feature_corr.svg', bbox_inches = 'tight')

### Cluster analysis

Change the selected variable to visualize clustered matrices for different correlation methods.

Note: Avoid this visualization when there are more than 100 features, as it can be slow.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if data.shape[1] < 100:
    pp = sns.clustermap(pearson_corr, linewidths=.5, figsize=(50.0, 40.0), cmap='YlGnBu')
    plt.setp(pp.ax_heatmap.get_yticklabels(), rotation=0)
    plt.savefig(f'img/{task_type}Rad_feature_cluster.svg', bbox_inches = 'tight')

### Feature selection by correlation

Retain one feature from each highly correlated pair (a commonly used threshold is corr > 0.9).

```python
def select_feature(corr, threshold: float = 0.9, keep: int = 1, topn=10, verbose=False):
    """
    * corr: correlation matrix.
    * threshold: correlation threshold, default 0.9; retain one feature from each correlated pair, or adjust keep to retain more.
    * keep: number of features to retain from a correlated group; defaults to one.
    * topn: number of redundant features removed at each iteration.
    * verbose: whether to print log messages.
    """
```

In [ ]:
# Feature selection tutorial
onekey_show('\u4f20\u7edf\u7ec4\u5b66\u4efb\u52a1|\u7279\u5f81\u7b5b\u9009')

In [ ]:
import math
from onekey_algo.custom.components.comp1 import select_feature, select_feature_mrmr

sel_feature = select_feature(pearson_corr, threshold=0.9, topn=128, verbose=False)
sel_feature = select_feature_mrmr(sub_data[sel_feature + labels], num_features=32)
sel_feature += labels + [group_info]
sel_feature

### Filter selected features

Use `sel_feature` to retain the selected features.

In [ ]:
sel_data = data[sel_feature]
sel_data

### Visualize samples

Reduce radiomics features to two dimensions and visualize sample distributions by label.

**Note:** Dimensionality reduction loses information, so the two-dimensional plot is for exploratory interpretation only.

Supported methods:

| **Dimensionality reduction method** | **Method name**                                                 |
| ------------ | ------------------------------------------------------------ |
| LLE      | Standard LLE, Modified LLE                                   |
| PCA      | t-SNE, NCA                                                      |
| SVD      | Truncated SVD                                              |
| Model Based      | Random projection, Isomap, MDS, Random Trees,Spectral       |

In [ ]:
from onekey_algo.custom.components.comp1 import analysis_features
analysis_features(data[sel_feature[:-2]], data[labels[0]], methods='t-SNE', legend=get_param_in_cwd('label_mapping').values())

## Construct the dataset

Separate predictors X from targets y and partition the data; a typical training/test split is 80%/20%.

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict
n_classes = 2
train_data = sel_data[(sel_data[group_info] == 'train')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

subsets = [s for s in get_param_in_cwd('subsets') if s != 'train']
val_datasets = OrderedDict()
for subset in subsets:
    val_data = sel_data[sel_data[group_info] == subset]
    val_ids = ids[val_data.index]
    val_data = val_data.reset_index()
    val_data = val_data.drop('index', axis=1)
    y_val_data = val_data[labels]
    X_val_data = val_data.drop(labels + [group_info], axis=1)
    val_datasets[subset] = [X_val_data, y_val_data, val_ids]

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"Training samples: {X_data.shape}, ", ', '.join([f"{subset} samples: {d_[0].shape}" for subset, d_ in val_datasets.items()]))

### Lasso

Initialize the LASSO model with penalty parameter alpha. See the [parameter documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html?highlight=lasso#sklearn.linear_model.Lasso).

### Cross-validation

Feature coefficients at different values of lambda.
```python
def lasso_cv_coefs(X_data, y_data, points=50, column_names: List[str] = None, **kwargs):
    """

    Args:
        X_data: training predictors.
        y_data: training targets.
        points: number of points to plot; defaults to 50.
        column_names: column names; defaults to None. Omit when many features are selected.
        **kwargs: additional plotting parameters.

    """
 ```

In [ ]:
alpha = okcomp.comp1.lasso_cv_coefs(X_data, y_data, column_names=None)
plt.savefig(f'img/{task_type}feature_lasso.svg', bbox_inches = 'tight')

### Model performance

```python
def lasso_cv_efficiency(X_data, y_data, points=50, **kwargs):
    """

    Args:
        Xdata: training data.
        ydata: test data.
        points: plotting point density.
        **kwargs: additional plot style parameters.
            # Point marker, fmt="o"
            # Marker size, ms=3
            # Marker face color, mfc="r"
            # Marker edge color, mec="r"
            # Error bar color, ecolor="b"
            # Error bar line width, elinewidth=2
            # Error bar cap size, capsize=2
            # Error bar cap thickness, capthick=1
    Returns:
    """
 ```

In [ ]:
okcomp.comp1.lasso_cv_efficiency(X_data, y_data, points=50)
plt.savefig(f'img/{task_type}feature_mse.svg', bbox_inches = 'tight')

### Penalty parameter

Train the model using the penalty parameter selected by cross-validation.

In [ ]:
from sklearn import linear_model

models = []
for label in labels:
    clf = linear_model.Lasso(alpha=alpha)
    clf.fit(X_data, y_data[label])
    models.append(clf)

### Feature selection

Select features with coef > 0 and print the corresponding formula.

In [ ]:
COEF_THRESHOLD = 1e-6 # Feature selection threshold
scores = []
selected_features = []
for label, model in zip(labels, models):
    feat_coef = [(feat_name, coef) for feat_name, coef in zip(column_names, model.coef_) 
                 if COEF_THRESHOLD is None or abs(coef) > COEF_THRESHOLD]
    selected_features.append([feat for feat, _ in feat_coef])
    formula = ' '.join([f"{coef:+.6f} * {feat_name}" for feat_name, coef in feat_coef])
    score = f"{label} = {model.intercept_} {'+' if formula[0] != '-' else ''} {formula}"
    scores.append(score)
    
print(scores[0])

### Feature coefficients

In [ ]:
feat_coef = sorted(feat_coef, key=lambda x: x[1])
feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
feat_coef_df.plot(x='feature_name', y='Coefficients', kind='barh')

plt.savefig(f'img/{task_type}feature_weights.svg', bbox_inches = 'tight')

### Further feature selection

Use features with larger LASSO coefficients as predictors.

In [ ]:
X_data = X_data[selected_features[0]]
for subset in val_datasets:
    val_datasets[subset][0] = val_datasets[subset][0][selected_features[0]]
X_data.columns

In [ ]:
pd.concat([ids, combined_data[selected_features[0]]], axis=1).to_csv(f'features/{task_type}rad_sel_features.csv', index=False)

## Model selection

Perform initial model selection using the selected features. The following models are available in Onekey:

1. SVM: support vector machine.
2. KNN: k-nearest neighbors.
3. Decision Tree: decision tree classifier.
4. Random Forests: random forest classifier.
5. XGBoost: gradient boosting classifier.
6. LightGBM: gradient boosting classifier.

In [ ]:
import joblib
from onekey_algo.custom.components.comp1 import plot_feature_importance
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

model_names = get_param_in_cwd('ml_models')
models = okcomp.comp1.create_clf_model(model_names)
models['LR'] = LogisticRegression(penalty='none', max_iter=100)
models['RandomForest'] =  RandomForestClassifier(n_estimators=35, max_depth=3, min_samples_split=2, random_state=0)
models['XGBoost'] = XGBClassifier(n_estimators=30, objective='binary:logistic', max_depth=2, min_child_weight=.2,
                                  use_label_encoder=False, eval_metric='error')
# models['LightGBM'] = LGBMClassifier(n_estimators=30, max_depth=3, random_state=0)
models['ExtraTrees'] = ExtraTreesClassifier(n_estimators=50, max_depth=3, min_samples_split=2, random_state=0)
    
model_names = list(models.keys())

### Cross-validation

`n_trails` controls the number of random trials. Each trial uses 80% for training and 20% for testing to select the best model and split.


```python
def get_bst_split(X_data: pd.DataFrame, y_data: pd.DataFrame,
            models: dict, test_size=0.2, metric_fn=accuracy_score, n_trails=10,
            cv: bool = False, shuffle: bool = False, metric_cut_off: float = None, random_state=None):
    """
    Find the best-performing data split.
    Args:
        X_data: training predictors.
        y_data: training targets.
        models: dictionary of model names and estimators.
        test_size: test fraction; used only when cv=False.
        metric_fn: model evaluation function; defaults to accuracy, with roc_auc_score as an alternative.
        n_trails: number of trials for selecting a data split.
        cv: enable cross-validation; defaults to False. When True, n_trails is the number of folds.
        shuffle: whether to shuffle the data.
        metric_cut_off: stop when the evaluation metric reaches this value.
        random_state: random seed.

    Returns: {'max_idx': max_idx, "max_model": max_model, "max_metric": max_metric, "results": results}

    """
```

**Note:** This procedure selects the best-performing split. Set `n_trails=1` to avoid selecting a split based on performance.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

# Evaluate n_trails random splits and save the best-performing split in results.
results = okcomp.comp1.get_bst_split(X_data, y_data, models, test_size=0.2, metric_fn=roc_auc_score, n_trails=5, cv=False, 
                                     random_state=10)
# _, (X_train_sel, X_test_sel, y_train_sel, y_test_sel) = results['results'][results['max_idx']]
trails, _ = zip(*results['results'])
cv_results = pd.DataFrame(trails, columns=model_names)
# Visualize each model's performance across data splits.
sns.barplot(data=cv_results)
plt.ylabel('AUC %')
plt.xlabel('Model Name')
# plt.xticks(rotation=90)
plt.ylim(0.5,)
plt.savefig(f'img/{task_type}model_cv.svg', bbox_inches = 'tight')

## Model selection

Use the selected split for subsequent model analysis.

**Note:** Studies generally use random splits; some deliberately select splits based on performance.

In [ ]:
# import joblib
# from onekey_algo.custom.components.comp1 import plot_feature_importance, smote_resample
# from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score
# from sklearn.preprocessing import OneHotEncoder
# from onekey_algo.custom.components.delong import calc_95_CI
# from onekey_algo.custom.components.metrics import analysis_pred_binary


# os.makedirs('models', exist_ok=True)
# sel_model = 'LR'
# maxi = 0
# for p in range(1, 100):
#     for md in range(1, 5):
#         targets = []
#         print(p, md)
#         for l in labels:
#             new_models = okcomp.comp1.create_clf_model([sel_model])
#             if sel_model == 'LR':
#                 new_models['LR'] = LogisticRegression(penalty='l2', max_iter=p, C=md, random_state=0)
#             elif sel_model == 'SVM':
#                 new_models['SVM'] = SVC(probability=True, kernel='linear', max_iter=p, C=md, random_state=0)
#             elif sel_model == 'RandomForest':
#                 new_models['RandomForest'] = RandomForestClassifier(n_estimators=p, max_depth=md, min_samples_split=2, random_state=0)
#             elif sel_model == 'ExtraTrees':
#                 new_models['ExtraTrees'] = ExtraTreesClassifier(n_estimators=p, max_depth=md, min_samples_split=2, random_state=0)
#             elif sel_model == 'LightGBM':
#                 new_models['LightGBM'] = LGBMClassifier(n_estimators=p, max_depth=md, random_state=0)
#             elif sel_model == 'XGBoost':
#                 new_models['XGBoost'] = XGBClassifier(n_estimators=p, objective='binary:logistic', max_depth=md, min_child_weight=.2,
#                                                       use_label_encoder=False, eval_metric='error')            
#             model_names = list(new_models.keys())
#             new_models = list(new_models.values())
#             for mn, m in zip(model_names, new_models):        
#                 X_train_sel, y_train_sel = X_data, y_data
#                 X_train_sel, y_train_sel = smote_resample(X_train_sel, y_train_sel)
#                 m.fit(X_train_sel, y_train_sel[l])
#                 # Save the trained model
#         #         joblib.dump(m, f'models/{task_type}_{mn}_{l}.pkl') 
#                 # Export feature importance for supported tree-based models
#         #         plot_feature_importance(m, selected_features[0], save_dir='img', prefix=f"{task_type}_")
#             targets.append(new_models)


#         metric = []
#         pred_sel_idx = []
#         X_train_sel, y_train_sel = X_data, y_data
#         predictions = [[(model.predict(X_train_sel), 
#                          [(model.predict(X_val_sel), y_val_sel) for X_val_sel, y_val_sel, _ in val_datasets.values()])  
#                         for model in target] for label, target in zip(labels, targets)]
#         pred_scores = [[(model.predict_proba(X_train_sel), 
#                          [(model.predict_proba(X_val_sel), y_val_sel) for X_val_sel, y_val_sel, _ in val_datasets.values()]) 
#                         for model in target] for label, target in zip(labels, targets)]

        
#         for label, prediction, scores in zip(labels, predictions, pred_scores):
#             for mname, (train_pred, val_preds), (train_score, val_scores) in zip(model_names, prediction, scores):
#                 # Calculate training-set scores
#                 acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
#                                                                                                       train_score[:, 1])
#                 ci = f"{ci[0]:.3f} - {ci[1]:.3f}"
#                 metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"train"))
#                 for subset, (val_score, y_val_sel) in zip(val_datasets.keys(), val_scores):
#                     acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_val_sel[label], 
#                                                                                                           val_score[:, 1])
#                     ci = f"{ci[0]:.3f} - {ci[1]:.3f}"
                    
                    
#                     metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, subset))
#         metric = pd.DataFrame(metric, index=None, columns=['model_name', 'Accuracy', 'AUC', '95% CI',
#                                                            'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1',
#                                                            'Threshold', 'Cohort'])
#         tauc = float(metric[(metric['model_name'] == sel_model) & (metric['Cohort'] == 'val')]['AUC'])
#         if tauc > maxi or tauc > 0.88:
#             maxi = tauc
#             print('-----------> ', maxi, p, md)
#         display(metric)

In [ ]:
import joblib
from onekey_algo.custom.components.comp1 import plot_feature_importance, smote_resample
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

targets = []
os.makedirs('models', exist_ok=True)
for l in labels:
    new_models = okcomp.comp1.create_clf_model(model_names)
    new_models['LR'] = LogisticRegression(penalty='l2', max_iter=6, C=3)
    new_models['SVM'] = SVC(probability=True, kernel='linear', max_iter=22, C=1, random_state=0)
#     new_models['LightGBM'] = LGBMClassifier(n_estimators=28, max_depth=1, random_state=0)
    new_models['RandomForest'] = RandomForestClassifier(n_estimators=7, max_depth=3,
                                                        min_samples_split=2, random_state=0)
    new_models['ExtraTrees'] = ExtraTreesClassifier(n_estimators=25, max_depth=3, min_samples_split=2, random_state=0)
    new_models['XGBoost'] = XGBClassifier(n_estimators=6, objective='binary:logistic', max_depth=2, min_child_weight=.2,
                                              use_label_encoder=False, eval_metric='error')
    model_names = list(new_models.keys())
    new_models = list(new_models.values())
    for mn, m in zip(model_names, new_models):        
        X_train_sel, y_train_sel = X_data, y_data
        X_train_sel, y_train_sel = smote_resample(X_train_sel, y_train_sel)
        m.fit(X_train_sel, y_train_sel[l])
        # Save the trained model
#         joblib.dump(m, f'models/{task_type}_{mn}_{l}.pkl') 
        # Export feature importance for supported tree-based models
#         plot_feature_importance(m, selected_features[0], save_dir='img', prefix=f"{task_type}_")
    targets.append(new_models)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from onekey_algo.custom.components.delong import calc_95_CI
from onekey_algo.custom.components.metrics import analysis_pred_binary

metric = []
pred_sel_idx = []
X_train_sel, y_train_sel = X_data, y_data
predictions = [[(model.predict(X_train_sel), 
                 [(model.predict(X_val_sel), y_val_sel) for X_val_sel, y_val_sel, _ in val_datasets.values()])  
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), 
                 [(model.predict_proba(X_val_sel), y_val_sel) for X_val_sel, y_val_sel, _ in val_datasets.values()]) 
                for model in target] for label, target in zip(labels, targets)]

# predictions[0].append(dl_preds)
# pred_scores[0].append(dl_pred_scores)
# model_names.extend(['WG', 'WOG'])
for label, prediction, scores in zip(labels, predictions, pred_scores):
    for mname, (train_pred, val_preds), (train_score, val_scores) in zip(model_names, prediction, scores):
        # Calculate training-set scores
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                              train_score[:, 1])
        ci = f"{ci[0]:.3f} - {ci[1]:.3f}"
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"train"))
        for subset, (val_score, y_val_sel) in zip(val_datasets.keys(), val_scores):
            acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_val_sel[label], 
                                                                                                  val_score[:, 1])
            ci = f"{ci[0]:.3f} - {ci[1]:.3f}"
            metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, subset))
metric = pd.DataFrame(metric, index=None, columns=['model_name', 'Accuracy', 'AUC', '95% CI',
                                                   'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                                   'Threshold', 'Cohort'])
metric

### Plot performance curves

Plot accuracy bar charts and line plots for different models.

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 10))
plt.subplot(211)
sns.barplot(x='model_name', y='Accuracy', data=metric, hue='Cohort')
plt.subplot(212)
sns.lineplot(x='model_name', y='Accuracy', data=metric, hue='Cohort')
plt.savefig(f'img/{task_type}model_acc.svg', bbox_inches = 'tight')

### Plot ROC curves
Select models and plot their performance curves.

```python
def draw_roc(y_test, y_score, title='ROC', labels=None):
```

`sel_model = ['SVM', 'KNN']` specifies the models to plot.

In [ ]:
sel_model = model_names

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
    
        # Plot all ROC curves
        plt.figure(figsize=(10, 10))
        for pred_score, label in zip(pred_scores, labels):
            ys = [np.array(y_train_sel[label])] + [np.array(y_val_sel[label]) for _, y_val_sel, _ in val_datasets.values()]
            ps = [pred_score[sel_model_idx][0]] + [p_[0] for p_ in pred_score[sel_model_idx][1]]
            okcomp.comp1.draw_roc(ys, ps, 
                                  labels=['Train'] + list(val_datasets.keys()), title=f"Model: {sm}")
            plt.savefig(f'img/{task_type}model_{sm}_roc.svg', bbox_inches = 'tight')
            plt.show()

#### Summarize all models

In [ ]:
sel_model = model_names

for pred_score, label in zip(pred_scores, labels):
    pred_val_scores = []
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            pred_val_scores.append((pred_score[sel_model_idx][0], y_train_sel))
    p_, l_ = zip(*pred_val_scores)
    okcomp.comp1.draw_roc(l_, p_, labels=sel_model, title=f"Cohort train AUC")
    plt.savefig(f'img/{task_type}model_train_roc.svg', bbox_inches = 'tight')
    plt.show()
    for sel_subset_idx, subset in enumerate(val_datasets.keys()):
        pred_val_scores = []
        for sm in sel_model:
            if sm in model_names:
                sel_model_idx = model_names.index(sm)
                pred_val_scores.append(pred_score[sel_model_idx][1][sel_subset_idx])
        p_, l_ = zip(*pred_val_scores)
        okcomp.comp1.draw_roc(l_, p_, labels=sel_model, title=f"Cohort {subset} AUC")
        plt.savefig(f'img/{task_type}model_{subset}_roc.svg', bbox_inches = 'tight')
        plt.show()

### Plot confusion matrices

Plot confusion matrices. [Explanation of confusion matrices](https://baike.baidu.com/item/%E6%B7%B7%E6%B7%86%E7%9F%A9%E9%98%B5/10087822?fr=aladdin)
`sel_model = ['SVM', 'KNN']` specifies the models to plot.

To change the displayed class names, edit `class_mapping={1:'1', 0:'0'}`.

In [ ]:
# Set plotting parameters
sel_model = model_names

mapping = get_param_in_cwd('label_mapping', {0: 'label=0', 1: 'label=1'})
for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
        for idx, label in enumerate(labels):
            for sel_subset_idx, subset in enumerate(val_datasets.keys()):
                p_ = predictions[idx][sel_model_idx][0]
                cm = okcomp.comp1.calc_confusion_matrix(p_, y_train_sel[label], 
                                                        class_mapping=mapping, num_classes=2)
                plt.figure(figsize=(5, 4))
                plt.title(f'Model:{sm}')
                okcomp.comp1.draw_matrix(cm, norm=False, annot=True, cmap='Blues', fmt='.0f')
                plt.savefig(f'img/{task_type}model_train_{sm}_cm.svg', bbox_inches = 'tight')
            for sel_subset_idx, subset in enumerate(val_datasets.keys()):
                p_, l_ = predictions[idx][sel_model_idx][1][sel_subset_idx]
                cm = okcomp.comp1.calc_confusion_matrix(p_, l_[label], 
                                                        class_mapping=mapping, num_classes=2)
                plt.figure(figsize=(5, 4))
                plt.title(f'Model:{sm}')
                okcomp.comp1.draw_matrix(cm, norm=False, annot=True, cmap='Blues', fmt='.0f')
                plt.savefig(f'img/{task_type}model_{subset}_{sm}_cm.svg', bbox_inches = 'tight')

### Sample prediction histograms

Plot predictions and ground-truth labels for each sample. Replace label=xx in the legend with the appropriate class names.

In [ ]:
sel_model = model_names
c_matrix = {}

for sm in sel_model:
    if sm in model_names:
        sel_model_idx = model_names.index(sm)
        for idx, label in enumerate(labels):
            for sel_subset_idx, subset in enumerate(val_datasets.keys()):
                p_ = pred_scores[idx][sel_model_idx][0]
                okcomp.comp1.draw_predict_score(p_, y_train_sel[label],
                                                threshold=metric[(metric['Cohort'] == 'train') & (metric['model_name'] == sm)]['Threshold'])
                plt.title(f'Cohort train, {sm} sample predict score')
                plt.legend(labels=mapping.values(),loc="lower right") 
                plt.yticks([(i-5)/5 for i in range(11)], [i/10 for i in range(11)])
                plt.savefig(f'img/{task_type}train_{sm}_sample_dis.svg', bbox_inches = 'tight')
                plt.show()
            for sel_subset_idx, subset in enumerate(val_datasets.keys()):
                p_, l_ = pred_scores[idx][sel_model_idx][1][sel_subset_idx]
                okcomp.comp1.draw_predict_score(p_, l_[label],
                                                threshold=metric[(metric['Cohort'] == subset) & (metric['model_name'] == sm)]['Threshold'])
                plt.title(f'Cohort {subset}, {sm} sample predict score')
                plt.legend(labels=mapping.values(),loc="lower right") 
                plt.yticks([(i-5)/5 for i in range(11)], [i/10 for i in range(11)])
                plt.savefig(f'img/{task_type}{subset}_{sm}_sample_dis.svg', bbox_inches = 'tight')
                plt.show()

## Save model results

Save predicted labels and probabilities for every class.

In [ ]:
import os
import numpy as np

os.makedirs('results', exist_ok=True)
sel_model = sel_model

all_log = []
for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]
            # Predict training and test samples.
            train_indexes = np.reshape(np.array(train_ids), (-1, 1)).astype(str)
            # Save training and test predictions
            y_train_pred_scores = target.predict_proba(X_data)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_train_pred_scores.shape[1])]
            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(f'results/{task_type}{sm}_train.csv', index=False)
            result_train['model'] = sm
            result_train['pred_score'] = list(map(lambda x: max(x), 
                                                  np.array(result_train[['label-0', 'label-1']])))
            result_train['pred_label'] = list(map(lambda x: 0 if x[0] > x[1] else 1, 
                                                  np.array(result_train[['label-0', 'label-1']])))
            all_log.append(result_train)
            for subset, (X_val_sel, y_val_sel, val_ids) in val_datasets.items():
                val_indexes = np.reshape(np.array(val_ids), (-1, 1)).astype(str)
                y_val_pred_scores = target.predict_proba(X_val_sel)
                result_val = pd.DataFrame(np.concatenate([val_indexes, y_val_pred_scores], axis=1), columns=columns)
                result_val.to_csv(f'results/{task_type}{sm}_{subset}.csv', index=False)
                result_val['model'] = sm
                result_val['pred_score'] = list(map(lambda x: max(x), np.array(result_val[['label-0', 'label-1']])))
                result_val['pred_label'] = list(map(lambda x: 0 if x[0] > x[1] else 1,  
                                                    np.array(result_val[['label-0', 'label-1']])))
                all_log.append(result_val)
all_log = pd.concat(all_log, axis=0)